# Reconstruct mesh from point cloud with meshlab

## Screened Poisoning Method (high computation more continous but more excessive faces)

In [ ]:
from pathlib import Path

import pymeshlab

path = Path("data/laso_grasp_only_affordance/mug/34869e23f9fdee027528ae0782b54aae/affordance.ply")

meshes = pymeshlab.MeshSet()
meshes.load_new_mesh(str(path))

# 1. Remeshing: Screened Poisson Reconstruction
meshes.generate_surface_reconstruction_screened_poisson(
    depth=8,
    fulldepth=5,
    samplespernode=2.0,
    pointweight=4.0,
)

# 2. Simplify reconstructed mesh
meshes.meshing_decimation_quadric_edge_collapse(
    targetfacenum=10000,
    preservenormal=True,
    preservetopology=True,
)

# 3. Save
output = path.with_name("visual.obj")
meshes.save_current_mesh(str(output))

print(f"Saved: {output}")
print(f"Vertices: {meshes.current_mesh().vertex_number()}")
print(f"Faces: {meshes.current_mesh().face_number()}")

Saved: data/laso_grasp_only_affordance/mug/34869e23f9fdee027528ae0782b54aae/visualize.obj
Vertices: 3415
Faces: 6768


In [ ]:
import trimesh

mesh = trimesh.Trimesh(
    vertices=meshes.current_mesh().vertex_matrix(),
    faces=meshes.current_mesh().face_matrix(),
    process=True,
)

output = path.with_name("visual.obj")
mesh.export(output)

'# https://github.com/mikedh/trimesh\nv -0.09336001 0.14727389 -0.59575837\nv 0.10778010 -0.38200140 -0.59575837\nv -0.09336001 -0.53168764 -0.59575837\nv -0.11311543 -0.38200140 -0.59575837\nv -0.09390320 0.14794115 -0.59575837\nv -0.12788663 -0.48798991 -0.59575837\nv 0.06562275 -0.52866354 -0.59575837\nv 0.10431123 -0.32900714 -0.59575837\nv -0.04036576 0.58736955 -0.59575837\nv -0.10278922 -0.32900714 -0.59575837\nv 0.04510166 0.09494689 -0.59575837\nv 0.01262850 -0.53936083 -0.59575837\nv 0.01488962 -0.27601289 -0.59575837\nv 0.01262850 0.04948522 -0.59575837\nv 0.01262850 -0.30515607 -0.59575837\nv -0.04379033 0.62488944 -0.59575837\nv -0.04036576 -0.54968568 -0.59575837\nv -0.09336001 -0.31542454 -0.59575837\nv 0.11861700 -0.51333604 -0.59575837\nv -0.01723005 0.09494689 -0.59575837\nv 0.09126775 -0.27601289 -0.59575837\nv 0.01262850 0.13389205 -0.59575837\nv -0.00881358 0.62488944 -0.59575837\nv -0.07340989 -0.54098416 -0.59575837\nv -0.04036576 -0.30898884 -0.59575837\nv -0.04

## Ball pivoting method (fast but does not ensure a fully connected mesh)

In [32]:
from pathlib import Path

def resolve_npz(data_root: Path, object_id: str | None, explicit: Path | None) -> Path:
    """Magic function to find your npz file with just object_id.
    
    Expect:
    ```
    object_id/
        |__affordance.npz
    ```
    """
    if not data_root.exists():
        raise ValueError("Invalid data root path!")
    if explicit is not None:
        path = explicit.resolve()
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    if not object_id:
        raise ValueError("Supply --object or --npz")

    matches = list(data_root.rglob(f"{object_id}/affordance.npz")) # search directly under data/
    if not len(matches): # between data and object id there is something else
        matches = list(data_root.rglob(f"*/{object_id}/affordance.npz"))
        if not len(matches):
            raise ValueError(f"Unable to resolve {object_id}/affordance.npz. Use --npz")
    
    return matches[0]

In [38]:
from pathlib import Path
from scipy.spatial import cKDTree
import numpy as np
import pymeshlab
# from generate_graspnet_grasp import resolve_npz
import os

DATA_ROOT = Path('data/laso_grasp_only_affordance').resolve()
npz_path = resolve_npz(DATA_ROOT, "mug/1bc5d303ff4d6e7e1113901b72a68e7c", None)
OUT_DIR = Path("outputs/reconstructed_mesh")
if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
out_path = os.path.join(OUT_DIR, 'mesh.ply')

with np.load(npz_path) as data:
    points = np.asarray(data["points"], dtype=np.float64)
    normals = np.asarray(data["normals"], dtype=np.float64)

cloud = pymeshlab.Mesh(
    vertex_matrix=points,
    v_normals_matrix=normals,
)

meshes = pymeshlab.MeshSet()
meshes.add_mesh(cloud, "point_cloud")

# LASO already provides normals.
# Recompute them only if the stored normals are unreliable.
# meshes.compute_normal_for_point_clouds(
#     k=20,
#     smoothiter=2,
# )



tree = cKDTree(points)

distances, _ = tree.query(points, k=2)

# Column zero is each point's distance to itself.
nearest_distances = distances[:, 1]

median_spacing = float(np.median(nearest_distances))
bbox_diagonal = float(np.linalg.norm(np.ptp(points, axis=0)))

alpha = 2.0

radius_percent = (
    100.0 * alpha * median_spacing / bbox_diagonal
)

print("median spacing:", median_spacing)
print("bounding-box diagonal:", bbox_diagonal)
print("suggested radius percentage:", radius_percent)

meshes.generate_surface_reconstruction_ball_pivoting(
    ballradius=pymeshlab.PercentageValue(radius_percent),
)
# radius_percentages = [1.5, 2.0, 2.5, 3.5]

# for index, radius in enumerate(radius_percentages):
#     meshes.generate_surface_reconstruction_ball_pivoting(
#         ballradius=pymeshlab.PercentageValue(radius),
#         deletefaces=index == 0,
#     )

# meshes.generate_surface_reconstruction_ball_pivoting(
#     # ballradius=pymeshlab.PercentageValue(2.5),
# )

# meshes.meshing_close_holes(
#     maxholesize=20,
#     selected=False,
#     newfaceselected=True,
#     selfintersection=True,
#     refinehole=True,
# )

# meshes.apply_coord_taubin_smoothing(
#     stepsmoothnum=10,
#     lambda_=0.5,
#     mu=-0.53,
# )

meshes.compute_normal_per_vertex()

meshes.save_current_mesh(str(out_path))

median spacing: 0.06087141077684127
bounding-box diagonal: 2.3268489051234162
suggested radius percentage: 5.232089685136874


In [39]:
from plotly import graph_objects 

mesh = meshes.current_mesh()

vertices = mesh.vertex_matrix()
faces = mesh.face_matrix()

print(f"vertices: {len(vertices)}")
print(f"faces: {len(faces)}")

if len(faces) == 0:
    raise RuntimeError("Surface reconstruction produced no triangle faces")

figure = graph_objects.Figure(
    data=[
        graph_objects.Mesh3d(
            x=vertices[:, 0],
            y=vertices[:, 1],
            z=vertices[:, 2],
            i=faces[:, 0],
            j=faces[:, 1],
            k=faces[:, 2],
            intensity=vertices[:, 2],
            colorscale="Viridis",
            flatshading=True,
            opacity=1.0,
            name="Reconstructed mesh",
        )
    ]
)

figure.update_layout(
    title="Ball-Pivoting Reconstruction",
    width=900,
    height=750,
    scene=dict(
        aspectmode="data",
        xaxis_title="X",
        yaxis_title="Y",
        zaxis_title="Z",
    ),
)

figure.show()

vertices: 2048
faces: 4003


In [40]:
import trimesh

mesh = trimesh.load(out_path, force="mesh", process=False)
components = mesh.split(only_watertight=False)

print("vertices:", len(mesh.vertices))
print("faces:", len(mesh.faces))
print("connected components:", len(components))
print("watertight:", mesh.is_watertight)

vertices: 2048
faces: 4003
connected components: 1
watertight: False
